In [ ]:
%matplotlib widget
import numpy as np
import scipy as sp
import itertools
import matplotlib as mpl
import matplotlib.pyplot as plt
import chemiscope

import ipywidgets
from ipywidgets import FloatSlider, IntSlider
import scwidgets
from scwidgets.check import (
    Check,
    CheckRegistry,
    assert_numpy_allclose,
    assert_numpy_floating_sub_dtype,
    assert_shape,
    assert_type,
)
from scwidgets.answer import AnswerRegistry
from scwidgets.code import ParameterPanel
from scwidgets.cue import CueOutput, CueObject, CueFigure
from scwidgets.exercise import CodeExercise, TextExercise

import ase
from ase.io import read, write

# to reuse the markdown texts, but we can also just convert them to html and throw away the dependency
import markdown

In [ ]:
answer_registry = AnswerRegistry(filename_prefix="module_02")
answer_registry

In [ ]:
check_registry = CheckRegistry()
check_registry

In [ ]:
module_summary = TextExercise(
    exercise_description="If you have general comments on this module",
    answer_registry=answer_registry,
    answer_key="Module summary",
)
module_summary

_Reference textbook / figure credits: Charles Kittel, "Introduction to solid-state physics", Chapter 1_

# Unit cells, fractional coordinates, lattices

A periodic structure is defined by a *lattice* and an a-periodic *repeat unit*. The lattice is a periodic set of points generated by all integer combinations of three *unit cell vectors* $\mathbf{a}_{1,2,3}$, i.e. 

$$
\mathbf{T} = u_1 \mathbf{a}_1 +  u_2 \mathbf{a}_2 +  u_3 \mathbf{a}_3, \quad u_{1,2,3} \in \mathbb{Z}
$$

The repeat unit, or *basis* is defined by the coordinates $\{x_i, y_i, z_i\}$ of a (usually small) number of atoms that sit in arbitrary positions within the unit cell. These are often given in *fractional coordinates* $\{s_{i1}, s_{i2}, s_{i3}\}$, such that the set of all atoms in the crystal is generated by 

$$
(u_1+s_{i1}) \mathbf{a}_1 +  (u_2+s_{i2}) \mathbf{a}_2 + (u_3+s_{i3}) \mathbf{a}_3
$$

with the $u_{1,2,3}$ ranging over all positive and negative integers, and $i$ ranging over the number of atoms in the basis.

There are 14 types of lattices known as _Bravais lattices_ in three dimensions, that are distinguished by the symmetry group of the lattice points. If you need to refresh your fundamentals of crystallography, and don't remember what a _face centered cubic_ or _body centered cubic_ lattice is, the [wikipedia page](https://en.wikipedia.org/wiki/Bravais_lattice) is excellent. Note also that the crystal structure (lattice+basis) can have more complicated symmetries, forming the 230 [space groups](https://en.wikipedia.org/wiki/Space_group)

Now, consider the structure below. This is just a finite set of atoms, and coordinates are listed individually. You can click on the atoms in the viewer and see its coordinates by clicking on the info panel below the viewer.

In [ ]:
positions = np.array(
    [
        [x, y, z]
        for x, y, z in itertools.product([0, 5, 10, 15], [0, 5, 10, 15], [0, 5, 10, 15])
    ]
)
ase_cube = ase.Atoms("Ga64", positions=positions)

properties = {}
properties["index"] = {
    "target": "atom",
    "values": list(range(1, len(ase_cube) + 1)),
}

properties["coordinates"] = {
    "target": "atom",
    "values": positions,
}

cs = chemiscope.show(
    [ase_cube],
    properties=properties,
    mode="structure",
    environments=chemiscope.all_atomic_environments([ase_cube], cutoff=40),
    settings={"structure": [{"spaceFilling": False, "environments": {"cutoff": 40}}]},
)
display(cs)

In [ ]:
def sc_lattice_basis():
    """
    Returns the lattice vectors and the basis (in fractional coordinates) that generates a simple cubic structure.

    :return: Three lattice vectors and a list of basis coordinates
    """
    # Write your solution, then click on the button below to update the plotter
    # and check against the reference value

    # a1 = []
    # a2 = []
    # a3 = []
    # basis = [[]]

    ### BEGIN SOLUTION
    a1 = [5, 0, 0]
    a2 = [0, 5, 0]
    a3 = [0, 0, 5]
    basis = [[0, 0, 0]]
    ### END SOLUTION
    return a1, a2, a3, basis


def reciprocal_lattice_vectors(a1, a2):
    # compute it in obfuscated way
    reciprocal_lattice = (
        2
        * np.pi
        * ase.Atoms(
            cell=[[a1[0], a1[1], 0], [a2[0], a2[1], 0], [0, 0, 1]]
        ).cell.reciprocal()
    )
    return reciprocal_lattice[0][:2], reciprocal_lattice[1][:2]


def ex01a_updater(code_exercise):
    a1, a2, a3, basis = code_exercise.run_code()
    positions = np.asarray(basis)
    h = np.asarray([a1, a2, a3])
    if h.size != 0:
        structure = ase.Atoms(
            "Ga" + str(len(basis)), positions=positions @ h.T, cell=[a1, a2, a3]
        )
        cue_output = code_exercise.cue_outputs[0]
        cue_output.display_object = chemiscope.show(
            frames=[structure],
            mode="structure",
            settings={
                "structure": [{"unitCell": True, "supercell": {"0": 3, "1": 3, "2": 3}}]
            },
        )


def ex01a_asserts(code_input_outputs: tuple) -> str:
    a1, a2, a3, basis = code_input_outputs
    if len(np.asarray(basis).shape) != 2:
        return (
            f"Basis should be a (n_atoms, 3) array, but I got {np.asarray(basis).shape}"
        )
    if np.asarray(basis).shape[1] != 3:
        return (
            f"Basis should be a (n_atoms, 3) array, but I got {np.asarray(basis).shape}"
        )
    if np.asarray(basis).shape[0] != 1:
        return f"The simple cubic lattice has usually a single-atom basis"


ex01a_description = """
Write a function that returns the lattice vectors and a basis that generates
the periodic structure that continues to infinity the motif above
<br><br>
<em>
NB: the validation code only tests for the primitive cell - you can come up
with correct structures that will be marked as incorrect. Trust your own
judgment (and look carefully at the visualization)
</em>'
"""

ex01a_exercise = CodeExercise(
    code=sc_lattice_basis,
    check_registry=check_registry,
    cue_outputs=[CueObject()],
    update_func=ex01a_updater,
    answer_key="ex01a",
    answer_registry=answer_registry,
    exercise_title="Exercise 01a - Lattice",
    exercise_description=ex01a_description,
)

check_registry.add_check(
    ex01a_exercise,
    asserts=[
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[{}],
    outputs_references=[([5, 0, 0], [0, 5, 0], [0, 0, 5], [[0, 0, 0]])],
)

display(ex01a_exercise)

In [ ]:
ex01b_description = """
Try to shift all the basis atom(s) by the same displacement.
Do you think that the resulting crystal would have different properties?"""
ex01b_answer = "Enter your answer here"

### BEGIN SOLUTION
ex01b_answer = """The periodic structure does not depend on the position of the single basis atom
in the unit cell. There are also other unit cells one could build that lead to
the same periodic structure."""
### END SOLUTION


ex01b_exercise = TextExercise(
    ex01b_answer,
    answer_registry=answer_registry,
    answer_key="ex01b",
    exercise_description=ex01b_description,
    exercise_title="Exercise 01b",
)
display(ex01b_exercise)

In [ ]:
ase_crystals = read("data/crystals.xyz", ":")

properties = {}
properties["lattice vectors a1"] = {
    "target": "structure",
    "values": np.asarray([str(list(a.cell[0])) for a in ase_crystals]),
}
properties["lattice vectors a2"] = {
    "target": "structure",
    "values": np.asarray([str(list(a.cell[1])) for a in ase_crystals]),
}
properties["lattice vectors a3"] = {
    "target": "structure",
    "values": np.asarray([str(list(a.cell[2])) for a in ase_crystals]),
}

properties["basis"] = {
    "target": "atom",
    "values": np.vstack([a.positions for a in ase_crystals]),
}


pb_crystal = chemiscope.show(
    ase_crystals,
    properties=properties,
    mode="structure",
    environments=chemiscope.all_atomic_environments(ase_crystals),
    settings={
        "structure": [
            {
                "bonds": True,
                "unitCell": True,
                "supercell": {"0": 3, "1": 3, "2": 3},
                "environments": {"cutoff": 6},
            }
        ]
    },
)
cue_output = CueOutput()
with cue_output:
    display(pb_crystal)


def pb_crystal_update(code_exercise):
    global pb_crystal
    cutoff = code_exercise.parameters["cutoff"]
    pb_crystal.settings = {"structure": [{"environments": {"cutoff": cutoff}}]}


pb_crystal_description = """
The following visualizer shows a collection of 9 crystal structures, containing
either one or two chemical species. Look at them, and try to understand what
type of lattice they belong to. You can visualize the environment of each atom
within an adjusable cutoff, which may help you appreciate the 3D nature of the
structure.
"""


pb_crystal_demo = CodeExercise(
    parameters={
        "cutoff": FloatSlider(
            value=6.0, min=1, max=12, step=0.1, description=r"cutoff / Å"
        )
    },
    cue_outputs=cue_output,
    update_func=pb_crystal_update,
    exercise_description=pb_crystal_description,
    exercise_title="Environment cutoff",
)
display(pb_crystal_demo)

In [ ]:
ex02_description = """
Each of the structures above are either face-centered (fcc), body-centered
(bcc) or simple cubic (sc). Write down in the box below what is the Bravais
lattice for each structure, and the size of the basis used to describe the
structure in each frame. </span>
<br><br>
<em> NB: pay attention: <br>
(1) you are asked about the symmetry of the
<b>lattice</b>: if you just look at the cell, you may be misled;<br>
(2) the number
of atoms included in the basis depend on the choice of cell.  </em>"""
ex02_answer = "Enter your answer here"

### BEGIN SOLUTION
ex02_answer = """
structure 1:  lattice: fcc, basis size: 4
structure 2: lattice: sc, basis size:1
structure 3: lattice: sc, basis size: 2
structure 4: lattice: sc, basis size: 4
structure 5: lattice: fcc, basis size: 8
structure 6: lattice: bcc, basis size: 2
structure 7: lattice: fcc, basis size: 2
structure 8: lattice: fcc, basis size: 1
structure 9: lattice: fcc, basis size: 4
"""
### END SOLUTION


ex02_exercise = TextExercise(
    ex02_answer,
    answer_registry=answer_registry,
    answer_key="ex02",
    exercise_description=ex02_description,
    exercise_title="Exercise 02",
)
display(ex02_exercise)

Using a supercell is not only a way of obtaining a more convenient/intuitive view of a crystal structure. It can also be useful to represent real materials, in which the ideal periodicity of the crystal is broken, e.g. because there are defects, interfaces, or simply disorder. In these cases, one often starts from a small unit cells and explicitly repeats it many times. _This is different from just replicating the unit cell for visualization purposes: the cell is also enlarged, and the coordinates of all atoms inside the unit cell can be adjusted independently_

ASE provides convenient utilities to replicate a structure. If `structure` is an `ase.Atoms` object, then
`structure.repeat((nx, ny, nz))` will replicate the structure the given number of times along each of the axes. You can learn more about the function using `help(ase.Atoms.repeat)`.

In [ ]:
help(ase.Atoms.repeat)

In [ ]:
def al_supercell(nrep: int):
    """
    Returns an ASE object describing a supercell built by replicating `nrep` times along each direction a conventional (4-atoms)
    fcc unit cell for aluminum. Use a lattice parameter of 4 Å.

    :param nrep: int: number of repetitions a supercell replicates

    :return: The replicated-cell ASE object
    """
    # Write your solution, then click on the button below to update the chemiscope viewer
    import ase

    al4 = ase.Atoms("Al4", pbc=True)
    # al4.cell = [ ... ]                 # lattice parameters of the conventional unit cell
    # al4.positions = [[atom1x, atom1y, atom1z],
    #                 [atom2x, atom2y, atom2z],
    #                 [atom3x, atom3y, atom3z],
    #                 [atom4x, atom4y, atom4z]] # positions of the 4 atoms in the conventional fcc cell

    # empty atom
    # al_replicated =  ...

    ### BEGIN SOLUTION
    al4 = ase.Atoms("Al4", pbc=True)
    al4.cell = [4, 4, 4]  # lattice parameters of the conventional unit cell
    al4.positions = [
        [0, 0, 0],
        [2, 2, 0],
        [2, 0, 2],
        [0, 2, 2],
    ]  # positions of the 4 atoms in the conventional fcc cell

    # empty atom
    al_replicated = al4.repeat(nrep)
    ### END SOLUTION

    return al_replicated


def ex03_updater(code_exercise):
    al_multi = code_exercise.run_code(**code_exercise.parameters)
    cue_object = code_exercise.cue_outputs[0]
    cue_object.display_object = chemiscope.show(
        frames=[al_multi],
        mode="structure",
        settings={"structure": [{"unitCell": True}]},
    )


def fingerprint_ase_atoms(ase_atom):
    return ase_atom.cell.array, np.sum(ase_atom.positions)


ex03_description = """
Write code to generate a supercell for <em>fcc</em> aluminum, using the number
of repeat units as an argument, and a lattice parameter of 4Å. You can use the
<code>repeat</code> ASE command, but if you feel adventurous you may as well
write a replicate function yourself. </span>
<br><br>
<em>
NB: the self-test function can generate false errors because it expects a
specific order of the basis atoms. Don't worry too much if it tells you one
test has not passed!
</em>
"""

ex03_exercise = CodeExercise(
    code=al_supercell,
    parameters={
        "nrep": IntSlider(
            value=2, min=1, max=4, step=1, description=r"$n_{\mathrm{repeat}}$"
        )
    },
    update_func=ex03_updater,
    update_mode="manual",
    cue_outputs=CueObject(),
    check_registry=check_registry,
    answer_key="ex03",
    answer_registry=answer_registry,
    exercise_title="Exercise 03 - Supercell",
    exercise_description=ex03_description,
)


position_ref_2 = np.array(
    [
        [0.0, 0.0, 0.0],
        [2.0, 2.0, 0.0],
        [2.0, 0.0, 2.0],
        [0.0, 2.0, 2.0],
        [0.0, 0.0, 4.0],
        [2.0, 2.0, 4.0],
        [2.0, 0.0, 6.0],
        [0.0, 2.0, 6.0],
        [0.0, 4.0, 0.0],
        [2.0, 6.0, 0.0],
        [2.0, 4.0, 2.0],
        [0.0, 6.0, 2.0],
        [0.0, 4.0, 4.0],
        [2.0, 6.0, 4.0],
        [2.0, 4.0, 6.0],
        [0.0, 6.0, 6.0],
        [4.0, 0.0, 0.0],
        [6.0, 2.0, 0.0],
        [6.0, 0.0, 2.0],
        [4.0, 2.0, 2.0],
        [4.0, 0.0, 4.0],
        [6.0, 2.0, 4.0],
        [6.0, 0.0, 6.0],
        [4.0, 2.0, 6.0],
        [4.0, 4.0, 0.0],
        [6.0, 6.0, 0.0],
        [6.0, 4.0, 2.0],
        [4.0, 6.0, 2.0],
        [4.0, 4.0, 4.0],
        [6.0, 6.0, 4.0],
        [6.0, 4.0, 6.0],
        [4.0, 6.0, 6.0],
    ]
)

check_registry.add_check(
    ex03_exercise,
    asserts=[
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[{"nrep": 1}, {"nrep": 2}],
    outputs_references=[
        (np.array([[4, 0, 0], [0, 4, 0], [0, 0, 4]]), 12.0),
        (np.array([[8, 0, 0], [0, 8, 0], [0, 0, 8]]), 288.0),
    ],
    fingerprint=fingerprint_ase_atoms,
    suppress_fingerprint_asserts=False,
)

display(ex03_exercise)

# Lattice planes and surfaces 

The general notation used to define lattice planes is somewhat contrived (because it actually relates to the reciprocal lattice!). A plane is defined by three points, so one can define a lattice plane by picking three points that are multiples of the unit cell vectors, $(n_1 \mathbf{a_1}, n_2 \mathbf{a_2}, n_3 \mathbf{a_3})$. The plane is determined by the reciprocals of the $n_{1,2,3}$, multiplied by their least common multiple so as to obtain three integer indices $(h k l)$. For instance, a plane that intercepts the lattice axes with $n_{1,2,3} = (4,3,1)$ has Miller indices $(3,4,12)$. _Only in a cubic lattice_ the plane identified by (h k l) has $h \mathbf{a_1} + k \mathbf{a_2} + l \mathbf{a_3}$ as a plane normal. 

If you need a more detailed discussion, see Chapter 1 of _Kittel_, or the [Wikipedia page on Miller indices](https://en.wikipedia.org/wiki/Miller_index) that has some good figures. 

In [ ]:
fcc_al = ase.lattice.cubic.FaceCenteredCubic("Al")
al_supercell = fcc_al.repeat((4, 4, 4))

normal_plane = np.array([1, 0, 0])
origin = np.diagonal(al_supercell.cell) / 2
distances = np.abs((al_supercell.positions - origin) @ normal_plane) / np.linalg.norm(
    normal_plane
)
atoms_on_plane = np.where(distances < 1e-5)[0]
al_supercell.numbers[atoms_on_plane] = 12


def update_surface_cut(code_exercise):
    h, k, l = code_exercise.parameters.values()
    al_supercell.numbers[:] = 13
    normal_plane = np.array([h, k, l])
    if np.linalg.norm(normal_plane) != 0:
        origin = np.diagonal(al_supercell.cell) / 2
        distances = np.abs(
            (al_supercell.positions - origin) @ normal_plane
        ) / np.linalg.norm(normal_plane)
        atoms_on_plane = np.where(distances < 1e-6)[0]
        al_supercell.numbers[atoms_on_plane] = 12

    cue_object = code_exercise.cue_outputs[0]
    cue_object.display_object = chemiscope.show(
        [al_supercell],
        mode="structure",
        settings={"structure": [{"spaceFilling": False, "unitCell": True}]},
    )


surface_cut_description = """
In the widget below, you can see a large supercell of <em>fcc</em> aluminum.
You can select different Miller indices, and see the atoms that belong to one
of the lattice planes highlighted in a different color. Experiment a bit and
note how e.g. <code>(2,2,4)</code> is equivalent to <code>(1,1,2)</code>, etc.
Note also how the density of atoms on low-index planes is higher than on
high-index planes.
"""

surface_cut_code_demo = CodeExercise(
    parameters={"h": (0, 4, 1), "k": (0, 4, 1), "l": (0, 4, 1)},
    cue_outputs=CueObject(),
    update_func=update_surface_cut,
    update_mode="manual",
    exercise_description=surface_cut_description,
)

display(surface_cut_code_demo)
surface_cut_code_demo.run_update()

In [ ]:
ex04_description = """
Activate the visualization of multiple replicas in the visualizer above (e.g.
<code>(2,2,2)</code>) by increasing the supercell replication counts along X, Y,
Z. Are the planes continuous across the edges of the supercell, if you pick a
<code>(1,0,0)</code> plane? And what happens if you pick <code>(1,1,1)</code>?
"""
ex04_answer = "Enter your answer here"

### BEGIN SOLUTION
ex04_answer = """The planes are continuous for (1,0,0), as well as for the other directions
aligned with the axes. When choosing another direction, there are gaps in the
surfaces when looked in a repeated cell geometry. This happens because the
equation that defines the plane is not necessarily consistent with the
translational symmetry. To obtain a continuous plane, one must choose a
supercell for which two of the axes span the desired lattice plane."""
### END SOLUTION


ex04_exercise = TextExercise(
    ex04_answer,
    answer_registry=answer_registry,
    answer_key="ex04",
    exercise_description=ex04_description,
    exercise_title="Exercise 04",
)
display(ex04_exercise)

Now let's create a supercell with an actual surface. While there are tools to do this for more complicated scenarios (see e.g. [here](https://wiki.fysik.dtu.dk/ase/ase/build/surface.html#create-specific-non-common-surfaces) for some examples using ASE), we will do this manually to understand better what is going on. 

If the surface is aligned with one of the lattice parameters, it is sufficient to first replicate the unit cell a number of times, and then artificially enlarge one of the edges of the unit cell. This will leave some empty space between the top and bottom layers of the structure, effectively leaving some atoms in contact with vacuum. 

<img src="figures/slab-100.png" width="400"/>

Note you always create _two_ surfaces, because you are still dealing with a periodic structure. This is usually referred to as a _slab geometry_. Note also that for complicated crystals with a basis, the unit cell can be cut at different positions, so that there can be multiple different surfaces corresponding to the same lattice direction, depending on the position of the cut. 

In [ ]:
def al_slab(nrep: int, nslab: int, vacuum: int):
    """
    Returns an ASE object describing Al (100) surfaces using a slab geometry with nrep x nrep x nslab cells,
    and a vacuum region along z. Use a lattice parameter of 4 Å.

    :param nrep: int:     number of repetitions in the x,y plane
    :param nslab: int:    number of repetitions in the z direction (which determines the slab thickness)
    :param vacuum: float: thickness of the vacuum layer (in Å) along the z direction

    :return: The ASE object describing the slab geometry
    """
    # Write your solution, then click on the button below to update the chemiscope viewer
    import ase

    al4 = ase.Atoms("Al4", pbc=True)
    # al4.cell = [ ... ] # lattice parameters of the conventional unit cell
    # al4.positions = [[, ,], [, ,], [, ,], [, ,]] # positions of the 4 atoms in the conventional fcc cell

    # bulk structure, replicated to obtain the desired slab size
    # al_replicated =  ...
    # create gap by adding some vacuum to the supercell in the z direction

    ### BEGIN SOLUTION
    al4 = ase.Atoms("Al4", pbc=True)
    al4.cell = [4, 4, 4]  # lattice parameters of the conventional unit cell
    al4.positions = [
        [0, 0, 0],
        [2, 2, 0],
        [2, 0, 2],
        [0, 2, 2],
    ]  # positions of the 4 atoms in the conventional fcc cell

    # empty atom
    al_replicated = al4.repeat((nrep, nrep, nslab))
    al_replicated.cell[2, 2] += vacuum
    ### END SOLUTION

    return al_replicated


def ex05_updater(code_exercise):
    al_multi = code_exercise.run_code(*code_exercise.parameters.values())
    code_exercise.cue_outputs[0].display_object = chemiscope.show(
        frames=[al_multi],
        mode="structure",
        settings={"structure": [{"unitCell": True}]},
    )


ex05_description = """
Write a function that creates a supercell for <em>fcc</em> aluminum with 4x4x2
replicas along the three directions. Use a conventional unit cell and a lattice
parameter of 4Å. Modify the <code>cell</code> of the resulting ASE structure to
add 20Å of vacuum along the $\\mathbf{a}_3$ direction, creating a slab geometry
with a surface along <code>(0,0,1)</code>
<br><br>
<em>NB: you can copy most of the code from exercise 03.</em>
"""

ex05_exercise = CodeExercise(
    code=al_slab,
    parameters={
        "nrep": IntSlider(
            value=4, min=1, max=6, step=1, description=r"$n_{\mathrm{rep}}$"
        ),
        "nslab": IntSlider(
            value=2, min=1, max=6, step=1, description=r"$n_{\mathrm{slab}}$"
        ),
        "vacuum": FloatSlider(
            value=20.0,
            min=0.0,
            max=40.0,
            step=0.5,
            description=r"$L_{\mathrm{vacuum}}$",
        ),
    },
    update_func=ex05_updater,
    update_mode="manual",
    cue_outputs=CueObject(),
    check_registry=check_registry,
    answer_key="ex05",
    answer_registry=answer_registry,
    exercise_title="Exercise 05",
    exercise_description=ex05_description,
)

check_registry.add_check(
    ex05_exercise,
    asserts=[
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[
        {"nrep": 1, "nslab": 1, "vacuum": 10},
        {"nrep": 4, "nslab": 2, "vacuum": 11},
    ],
    outputs_references=[
        (np.array([[4.0, 0.0, 0.0], [0.0, 4.0, 0.0], [0.0, 0.0, 14.0]]), 12.0),
        (np.array([[16.0, 0.0, 0.0], [0.0, 16.0, 0.0], [0.0, 0.0, 19.0]]), 2176.0),
    ],
    fingerprint=fingerprint_ase_atoms,
    suppress_fingerprint_asserts=False,
)

ex05_exercise

Creating a `(1,1,1)` surface is somewhat more complicated, because the conventional unit cell is not aligned properly. One way to create a slab with the appropriate orientation is to create a _primitive_ face-centered cell and to create the slab by _rescaling_ one of the cell vectors. 

In [ ]:
def al_slab_111(nrep: int, nslab: int, zscale: float):
    """
    Returns an ASE object describing Al (111) surfaces using a slab geometry with
    nrep x nrep x nslab copies of the primitive cell. Vacuum must be created by
    elongating one of the lattice vectors by a factor zscale. Use a lattice parameter of 4 Å.


    :param nrep: int:     number of repetitions in the x,y plane
    :param nslab: int:    number of repetitions in the z direction (which determines the slab thickness)
    :param zscale: float: multiplicative factors by which the third lattice vector should be magnified to create some vacuum

    :return: The ASE object describing the slab geometry
    """
    # Write your solution, then click on the button below to update the chemiscope viewer
    import ase

    al = ase.Atoms("Al", pbc=True)
    # al.cell = [ ... ] # lattice parameters of the *primitive* unit cell
    al.positions = [[0, 0, 0]]  # just one-atom basis!

    # empty atom
    # al_replicated =  ...
    # create gap

    ### BEGIN SOLUTION
    al = ase.Atoms("Al", pbc=True)
    al.cell = [
        [2, 2, 0],
        [2, 0, 2],
        [0, 2, 2],
    ]  # lattice parameters of the *primitive* unit cell
    al.positions = [[0, 0, 0]]  # just one-atom basis!

    # empty atom
    al_replicated = al.repeat((nrep, nrep, nslab))
    al_replicated.cell[2] *= zscale
    # create gap
    ### END SOLUTION

    return al_replicated


def ex06_updater(code_exercise):
    al_multi = code_exercise.run_code(**code_exercise.parameters)
    code_exercise.cue_outputs[0].display_object = chemiscope.show(
        frames=[al_multi],
        mode="structure",
        settings={"structure": [{"unitCell": True}]},
    )


ex06_parameters = dict(
    nrep=IntSlider(4, 1, 6, 1, description=r"$n_{\mathrm{rep}}$"),
    nslab=IntSlider(2, 1, 6, 1, description=r"$n_{\mathrm{slab}}$"),
    zscale=FloatSlider(
        4.0, min=1.0, max=10.0, step=0.5, description=r"$z_{\mathrm{scale}}$"
    ),
)

ex06_description = """
Write a function that creates a supercell for <em>fcc</em> aluminum, starting
with the <b>primitive</b> cell replicated as 8x8x2 along the lattice vectors.
Then <em>multiply</em>_ $\\mathbf{a}_3$ by a scaling factor to create the
surface. Visualize the resulting structure
"""

ex06_exercise = CodeExercise(
    code=al_slab_111,
    parameters=ex06_parameters,
    cue_outputs=CueOutput(),
    update_func=ex06_updater,
    update_mode="manual",
    check_registry=check_registry,
    answer_registry=answer_registry,
    answer_key="ex06",
    exercise_description=ex06_description,
    exercise_title="Exercise 06",
)


check_registry.add_check(
    ex06_exercise,
    asserts=[
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[
        {"nrep": 1, "nslab": 1, "zscale": 1},
        {"nrep": 1, "nslab": 2, "zscale": 2},
    ],
    outputs_references=[
        (np.array([[2, 2, 0], [2, 0, 2], [0, 2, 2]]), 0.0),
        (np.array([[2, 2, 0], [2, 0, 2], [0, 8, 8]]), 4.0),
    ],
    fingerprint=fingerprint_ase_atoms,
    suppress_fingerprint_asserts=False,
)
display(ex06_exercise)

In [ ]:
ex07_description = """
What is the direction of the primitive lattice parameters relative to the
conventional <em>fcc</em>_ index system? Is the surface normal parallel to the
$\\mathbf{a}_3$ cell vector?  In order to create an easier to visualize
structure, one often builds a unit cell that is orthorhombic and has one axis
that is parallel to the surface normal. Can you come up with three
low-Miller-indices directions that are mutually orthogonal and could be used to
define an appropriate unit cell?
"""
ex07_answer = "Enter your answer here"

### BEGIN SOLUTION
ex07_answer = """
The primitive lattice vectors are aligned along the (110), (101) and (011)
directions of the conventional fcc cell. Thus, the surface normal (111)  is not
parallel to the a3 vector. To have an orthorhombic cell with one axis aligned
along (111) one would need to pick the other two vectors to be mutually
orthogonal to this, e.g. (0 1-1) and (-2 1 1). 
"""
### END SOLUTION


ex07_exercise = TextExercise(
    ex07_answer,
    answer_registry=answer_registry,
    answer_key="ex07",
    exercise_description=ex07_description,
    exercise_title="Exercise 07",
)
display(ex07_exercise)

# Reciprocal lattice in 2D

From here on, we work exclusively with 2D lattices,

$$
\mathbf{T} = u_1 \mathbf{a}_1 + u_2 \mathbf{a}_2, \quad \mathbf{a}_{1,2} \in \mathbb{R}^2, u_{1,2} \in Z
$$

because they allow for simpler visualization of the core concepts. Most of the concepts we develop and demonstrate, however, apply equally well to actual 3D systems. 

The idea of the reciprocal lattice is deeply linked to the idea of scattering, and constructive interference. A plane wave $e^{\mathrm{i} \mathbf{k}\cdot \mathbf{x}}$ originating at the origin will be in phase with similar waves originating at all other lattice points if its wavevector $\mathbf{k}$ satisfies the phase relation $\mathbf{k}\cdot \mathbf{T} = 2\pi n$, where $n$ is an integer, for each and every lattice vector. 

We can build a _reciprocal lattice_ that consist of integer combinations of basis vectors $\mathbf{b}_{1,2}$

$$
\mathbf{G} = v_1 \mathbf{b}_1 + v_2 \mathbf{b}_2, \quad \mathbf{b}_{1,2} \in \mathbb{R}^2, v_{1,2} \in Z
$$

such that every reciprocal lattice point is a wavevector that results in a complete in-phase scattering from the direct lattice. To do this, we need relationships to hold between the lattice vectors: $\mathbf{b}_{1,2} \cdot \mathbf{a}_{1,2} = 2\pi$, $\mathbf{b}_{1,2}\cdot \mathbf{a}_{2,1} = 0$. Thus, 
$\mathbf{b}_1$ must be orthogonal to $\mathbf{a}_2$ and $\mathbf{b}_2$ must be orthogonal to $\mathbf{a}_1$.

We can achieve this (including also normalization that ensure the correct scaling $\mathbf{b}_{1,2} \cdot \mathbf{a}_{1,2} = 2\pi$, with the definition

$$ \mathbf{b}_1 = 2\pi\frac{\mathbf{R}\mathbf{a}_2}{\mathbf{a}_1\cdot\mathbf{R}\mathbf{a}_2}, \quad \mathbf{b}_2 = 2\pi\frac{\mathbf{R}\mathbf{a}_1}{\mathbf{a}_2\cdot\mathbf{R}\mathbf{a}_1},\quad\quad\mathbf{R} = \begin{bmatrix}0 & -1\\ 1 & 0\end{bmatrix}$$

where $\mathbf{R}$ is a $\pi/2$ rotation. 

In [ ]:
ex08_description = """
What are the reciprocal lattice vectors for a 2D rectangular Bravais lattice
with lattice vectors $(1,0)$ and $(0,2)$?
"""
ex08_answer = "Enter your answer here"

### BEGIN SOLUTION
ex08_answer = """
The reciprocal primitive vectors are:
 b_1 = [2*pi, 0] b_2 = [0, pi]
"""
### END SOLUTION


ex08_exercise = TextExercise(
    ex08_answer,
    answer_registry=answer_registry,
    answer_key="ex08",
    exercise_description=ex08_description,
    exercise_title="Exercise 08",
)
display(ex08_exercise)

In [ ]:
from typing import Tuple


def reciprocal_lattice(a1: list, a2: list) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return the 2D reciprocal unit cell vectors.

    :param a1: list: unit cell vector a1 [a1x, a1y]
    :param a2: list: unit cell vector a2 [a2x, a2y]

    :return: reciprocal lattice unit cell vectors
    """
    import numpy as np
    from numpy import pi

    # this is just to accept all sort of sequence inputs
    a1 = np.asarray(a1)
    a2 = np.asarray(a2)

    # matrix-matrix and matrix-vector multiplication can be achieved with the @ operator
    # For example `R @ a` multiplies the matrix R with the vector a
    R = np.array([[0, -1], [1, 0]])  # <- this converts a nested list to a numpy array

    # change to the correct expression
    b1 = 2 * pi * a1
    b2 = 2 * pi * a2

    ### BEGIN SOLUTION
    # change to the correct expression
    b1 = 2 * pi * a2 @ R / (a2 @ R @ a1)
    b2 = 2 * pi * a1 @ R / (a1 @ R @ a2)
    ### END SOLUTION

    return np.asarray(b1), np.asarray(
        b2
    )  # this is just to return a standard format even if you define b1 and b2 as another type of iterable


def plot_lattice(
    ax,
    a1,
    a2,
    basis=None,
    alphas=None,
    s=20,
    c="red",
    lattice_size=60,
    head_length=0.5,
    head_width=0.2,
    width=0.05,
):
    if basis is None:
        basis = np.array([[0, 0]])
    A = np.array([a1, a2])
    # each atom in the basis gets a different basis alpha value when plotted
    if alphas is None:
        alphas = np.linspace(1, 0.3, len(basis))
    for i in range(len(basis)):
        lattice = (np.mgrid[:lattice_size, :lattice_size].T @ A + basis[i]).reshape(
            -1, 2
        )
        lattice -= (np.array([lattice_size // 2, lattice_size // 2]) @ A).reshape(-1, 2)
        ax.scatter(lattice[:, 0], lattice[:, 1], color=c, s=s, alpha=alphas[i])

    ax.fill(
        [0, a1[0], (a1 + a2)[0], a2[0]],
        [0, a1[1], (a1 + a2)[1], a2[1]],
        color=c,
        alpha=0.2,
    )
    ax.arrow(
        0, 0, a1[0], a1[1], width=width, length_includes_head=True, fc=c, ec="black"
    )
    ax.arrow(
        0, 0, a2[0], a2[1], width=width, length_includes_head=True, fc=c, ec="black"
    )


def plot_reciprocal_and_real_lattice(axes, a11, a12, a21, a22, basis=None):
    if basis is None:
        basis = np.array([[0, 0]])
    a1 = np.array([a11, a12])
    a2 = np.array([a21, a22])
    b1, b2 = ex09_exercise.run_code(a1, a2)

    plot_lattice(axes[0], a1, a2, basis, s=20, c="red")
    plot_lattice(axes[1], b1 / (2 * np.pi), b2 / (2 * np.pi), None, s=20, c="blue")

    axes[0].set_title("real space")
    axes[0].set_xlim(-5, 5)
    axes[0].set_ylim(-5, 5)
    axes[0].set_xlabel("$x$ / Å")
    axes[0].set_ylabel("$y$ / Å")

    axes[1].set_title("reciprocal space")
    axes[1].set_xlim(-5, 5)
    axes[1].set_ylim(-5, 5)
    axes[1].set_xlabel("$k_x/2\pi$ / Å$^{-1}$")
    axes[1].set_ylabel("$k_y/2\pi$ / Å$^{-1}$")


def ex09_updater(code_exercise: CodeExercise):
    cue_figure = code_exercise.cue_outputs[0]
    axes = cue_figure.figure.get_axes()
    basis = np.array([[0, 0]])
    a11, a12, a21, a22 = code_exercise.parameters.values()
    plot_reciprocal_and_real_lattice(axes, a11, a12, a21, a22, basis)


ex09_figure, _ = plt.subplots(1, 2, figsize=(7.5, 3.8), tight_layout=True)
ex09_figure.canvas.header_visible = False
ex09_figure.canvas.footer_visible = False
ex09_figure.canvas.toolbar_visible = False
ex09_figure.canvas.resizable = False
ex09_cue_figure = CueFigure(ex09_figure)

ex09_parameters = dict(
    a11=FloatSlider(1.0, min=-4, max=4, step=0.1, description=r"$a_{11} / Å$"),
    a12=FloatSlider(0.0, min=-4, max=4, step=0.1, description=r"$a_{12} / Å$"),
    a21=FloatSlider(0.0, min=-4, max=4, step=0.1, description=r"$a_{21} / Å$"),
    a22=FloatSlider(2.0, min=-4, max=4, step=0.1, description=r"$a_{22} / Å$"),
)


ex09_description = """
Write a function that computes the reciprocal lattice vectors given
$\mathbf{a}_{1,2}$. Use the demo widget to experiment and get an intuitive
understanding of what's going on.
"""

ex09_exercise = CodeExercise(
    code=reciprocal_lattice,
    parameters=ex09_parameters,
    cue_outputs=ex09_cue_figure,
    update_func=ex09_updater,
    update_mode="continuous",
    check_registry=check_registry,
    answer_registry=answer_registry,
    answer_key="ex09",
    exercise_description=ex09_description,
    exercise_title="Exercise 09",
)


ex09_inputs_parameters = [
    {"a1": (0, 1), "a2": (1, 0)},
    {"a1": (1, 1), "a2": (1, -1)},
    {"a1": (0, 2), "a2": (2, 1)},
]

check_registry.add_check(
    ex09_exercise,
    asserts=[
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[
        {"a1": [0, 1], "a2": [1, 0]},
        {"a1": [1, 1], "a2": [1, -1]},
        {"a1": [0, 2], "a2": [2, 1]},
    ],
    outputs_references=[
        reciprocal_lattice_vectors(inputs["a1"], inputs["a2"])
        for inputs in ex09_inputs_parameters
    ],
)

display(ex09_exercise)
ex09_exercise.run_update()

# Diffraction from a lattice

To understand diffraction, you need to consider that scattering is a process involving an incoming wave, $e^{-\mathrm{i} \mathbf{k}_{\mathrm{in}} \cdot \mathbf{x}}$. Each lattice point will be hit by the wave with a different phase 
$e^{-\mathrm{i} \mathbf{k}_{\mathrm{in}} \cdot \mathbf{T}}$ and act as a spherical scatterer. The detector will be in a given position $\mathbf{R}_d$, and if it is sufficiently far it will pick up a plane wave in that direction, with outgoing wavevector $\mathbf{k}_{\mathrm{out}}$. Depending on the position of the scattering point, it will accumulate a further phase $e^{\mathrm{i} \mathbf{k}_{\mathrm{out}} \cdot \mathbf{T}}$ (the change in sign coming from the fact the distance traveled will be $~(\mathbf{R}_d - \mathbf{T})$. The scheme below, drawn for arbitrary position of the scattering centers, applies clearly also to the case when scattering occurs from lattice points

<img src="figures/scattering.png" width="600"/>

The condition to have constructive interference is therefore to have the scattering vector $\mathbf{k}=\mathbf{k}_\mathrm{in}-\mathbf{k}_\mathrm{out}$ be a vector of the reciprocal lattice.

For typical diffraction experiments, the scattering process is elastic, meaning that the modulus of the wavevector does not change during the interaction, so $|\mathbf{k}_{\mathrm{in}}|=|\mathbf{k}_{\mathrm{out}}|$. Furthermore, the modulus of the wavevector depends on the source of radiation used in the experiment. 

In [ ]:
ex10_description = """
What are the smallest and largest reciprocal lattice vectors that can be
observed with electromagnetic radiation with wavelength $\lambda$? Think of the
geometry of the experiment, and take the modulus of the incoming and outgoing
wavevectors to be $2\pi/\lambda$. 
"""
ex10_answer = "Enter your answer here"

### BEGIN SOLUTION
ex10_answer = """
The smallest reciprocal lattice vector is always the (0,0,0) vector -
corresponding to forward scattering direction where k_in is parallel to k_out.
The largest vector corresponds to backscattering, with k_out=-k_in. The largest
G vector that can be recorded has a modulus of 4pi/lambda 
"""
### END SOLUTION


ex10_exercise = TextExercise(
    ex10_answer,
    answer_registry=answer_registry,
    answer_key="ex10",
    exercise_description=ex10_description,
    exercise_title="Exercise 10",
)
display(ex10_exercise)

The widget below allows you to experiment with the geometry of a scattering experiment. You can define a direct lattice, the wavelength of the incoming light, and the angle between incoming and outgoing beam. 
The scheme that is drawn around the origin is the construction for the Ewald circle (2D equivalent of the [Ewald sphere](https://en.wikipedia.org/wiki/Ewald%27s_sphere)). There may be scattering only if the scattering vector $\mathbf{k}$ coincides with a point of the reciprocal lattice. 

In [ ]:
def ex10_updater(code_exercise: CodeExercise):
    cue_figure = code_exercise.cue_outputs[0]
    a11, a12, a21, a22, phi, wavelength, two_theta = code_exercise.parameters.values()
    axes = cue_figure.figure.get_axes()

    def rot2d(angle):
        return np.array(
            [[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]]
        )

    rot_phi = rot2d(phi)
    a1 = np.array([a11, a12]) @ rot_phi
    a2 = np.array([a21, a22]) @ rot_phi

    # reuse
    plot_reciprocal_and_real_lattice(axes, a1[0], a1[1], a2[0], a2[1])

    head_length = 0.5
    head_width = 0.2
    width = 0.05
    lw = 1
    alpha_unit_cell_vectors = 0.8

    k_in = (
        2 * np.pi * np.array([-1, 0]) / wavelength / (2 * np.pi)
    )  # <- scale by 2pi so it is consistent with reciprocal lattice plot
    k_out = rot2d(two_theta) @ k_in
    k = k_in - k_out
    scatter_origin = np.array([0, 0])
    axes[1].arrow(
        scatter_origin[0] + k_in[0],
        scatter_origin[1] + k_in[1],
        -k_in[0],
        -k_in[1],
        lw=lw,
        head_width=head_width,
        head_length=head_length,
        width=0.05,
        length_includes_head=True,
        fc="red",
        ec="red",
        label="$k_{in}}$",
    )
    axes[1].arrow(
        scatter_origin[0] + k_in[0],
        scatter_origin[1] + k_in[1],
        -k_out[0],
        -k_out[1],
        lw=lw,
        head_width=head_width,
        head_length=head_length,
        width=0.05,
        length_includes_head=True,
        fc="orange",
        ec="orange",
        label="$k_{out}$",
    )
    axes[1].arrow(
        scatter_origin[0],
        scatter_origin[1],
        k[0],
        k[1],
        lw=lw,
        head_width=head_width,
        head_length=head_length,
        width=0.05,
        length_includes_head=True,
        fc="black",
        ec="black",
        label="$k$",
    )

    circle_theta = np.linspace(-np.pi, np.pi, 200)
    r = 2 * np.pi / wavelength / (2 * np.pi)
    axes[1].plot(
        np.sin(circle_theta) * r + scatter_origin[0] + k_in[0],
        np.cos(circle_theta) * r + scatter_origin[1] + k_in[1],
        color="red",
        label="Ewald circle",
    )
    r = np.linalg.norm(k)
    axes[1].plot(
        np.sin(circle_theta) * r + scatter_origin[0],
        np.cos(circle_theta) * r + scatter_origin[1],
        color="black",
        label="$|\mathbf{k}|$ circle",
    )
    axes[1].legend(loc="lower right")


ex10_figure, _ = plt.subplots(1, 2, figsize=(7.5, 3.8), tight_layout=True)
ex10_figure.canvas.header_visible = False
ex10_figure.canvas.footer_visible = False
ex10_figure.canvas.toolbar_visible = False
ex10_figure.canvas.resizable = False
ex10_cue_figure = CueFigure(ex10_figure)


ex10_parameters = dict(
    a11=FloatSlider(1.0, min=-4, max=4, step=0.1, description=r"$a_{11} / Å$"),
    a12=FloatSlider(0.0, min=-4, max=4, step=0.1, description=r"$a_{12} / Å$"),
    a21=FloatSlider(0.0, min=-4, max=4, step=0.1, description=r"$a_{21} / Å$"),
    a22=FloatSlider(2.0, min=-4, max=4, step=0.1, description=r"$a_{22} / Å$"),
    phi=FloatSlider(0.0, min=0.0, max=2 * np.pi, step=0.05, description=r"$\phi$"),
    wavelength=FloatSlider(
        1.0, min=0.2, max=2, step=0.05, description=r"$\lambda$ / Å"
    ),
    two_theta=FloatSlider(1.0, min=0.1, max=np.pi, step=0.05, description=r"$2\theta$"),
)

# we need to reload the widget  when the solution of ex09
# computing the reciprocal unit cell vectors has been updated

ex10_code_demo = CodeExercise(
    parameters=ex10_parameters,
    cue_outputs=ex10_cue_figure,
    update_func=ex10_updater,
    update_mode="continuous",
)
display(ex10_code_demo)
ex10_code_demo.run_update()

You will notice that it is not easy, with a fixed position of the incoming light, to orient the detector so that $\mathbf{k}$ corresponds to a reciprocal lattice vector. The widget above allows you to change the orientation of the direct lattice. Note how, by changing the orientation of the crystal, you can bring any of the reciprocal lattice point that lie on the circle with radius $k$ in coincidence with the scattering wavevector $\mathbf{k}$.

In [ ]:
ex11_description = """
What happens if you consider scattering from a collection of crystals with
different, random orientations (a powder sample)? Does diffraction depend on
the orientation of the sample?
"""
ex11_answer = "Enter your answer here"

### BEGIN SOLUTION
ex11_answer = """
In a powder sample tiny crystals are oriented in all possible directions. This
is equivalent to integrating over the angle phi, so that the orientation of the
sample (and of *G*) does not matter, but only its modulus."""
### END SOLUTION


ex11_exercise = TextExercise(
    ex11_answer,
    answer_registry=answer_registry,
    answer_key="ex11",
    exercise_description=ex11_description,
    exercise_title="Exercise 11",
)
display(ex11_exercise)

# Diffraction from an atomic structure

An actual crystalline structure involves both a lattice and of an a-periodic basis $\{\mathbf{s}_i\}$ of atoms. Each will scatter with a form factor $f_i$ (that depends on the modulus of $\mathbf{k}$ but we will take to be a constant proportional to the atomic charge $Z_i$). The scattered amplitude can be written as  

$$
F(\mathbf{k}) = \frac{1}{N} \sum^N_{j \textrm{ atom}} f_j\exp(-\mathrm{i}\mathbf{k}\cdot\mathbf{r}_j)\textrm{, with }\mathbf{r}_j\textrm{ position of atom }j
$$

by breaking the position of atoms into lattice vector and basis positions ($\mathbf{r}_j=\mathbf{T}+\mathbf{s}_m$), we can write 

$$
F(\mathbf{k}) = \frac{1}{n_\mathrm{cell} n_\mathrm{basis}} 
\sum^{n_\mathrm{cell}}_{\mathbf{T}} \exp(-\mathrm{i}\mathbf{k}\cdot \mathbf{T}) 
\sum_m^{n_\mathrm{basis}} f_m \exp(-\mathrm{i}\mathbf{k}\cdot\mathbf{s}_m).
$$

One sees that the sum over lattice vectors selects the diffracted wavevectors that correspond to a reciprocal lattice vector, $\mathbf{k}=\mathbf{G}$, as we saw in the previous section. The sum over the atomic basis modulates the amplitude of the peak through a _structure factor_, which we will see encodes information on the type and position of atoms within the unit cell

$$
F_\mathbf{G} = \sum_m^{n_\mathrm{basis}} f_m \exp(-\mathrm{i}\mathbf{G}\cdot\mathbf{s}_m).
$$



<img src="figures/diffraction.png" width="600"/>

In the scattering geometry above, one can determine the conditions for scattering in terms of the scattering angle $2\theta$, as follows. If $k$ is the modulus of both $\mathbf{k}_\mathrm{in}$ and $\mathbf{k}_\mathrm{out}$, and $\mathbf{G} = \mathbf{k} = \mathbf{k}_\mathrm{in}-\mathbf{k}_\mathrm{out}$ must hold, a first necessary condition is

$$
G^2 = |\mathbf{k}_\mathrm{in}-\mathbf{k}_\mathrm{out}|^2 = 2k^2 (1-\cos 2\theta) = 4k^2 sin^2 \theta
$$

hence, $\sin\theta=G/2k$. To determine the orientation of $\mathbf{G}$ a second condition is needed. We can consider the angle between $\mathbf{G}$ and the incoming vector $\phi$, that can be set by changing the orientation of the crystal. Thus, by writing $|\mathbf{G}-\mathbf{k}_\mathrm{in}|^2 = |\mathbf{k}_\mathrm{out}|^2$ we can easily get $G = 2 k \cos\phi$, and hence $\cos\phi = \sin\theta$. If the sample is formed by uniformly oriented grains, the second condition is always satisfied by some crystals, and so the diffraction pattern is just a sequence of peaks at particular values of the angle $2\theta$.  The intensity of each peak is given by the square modulus of the structure factor, $|F_\mathbf{G}|^2$.

This widget below computes the powder (rotationally averaged) diffraction pattern for a crystal with a basis of two atoms. The function computes the list of diffraction peaks, with the corresponding structure factor. 
It is not entirely trivial, and so it is already implemented: you don't have to change it, but you can read it and understand what it does. Experiment with the widget, and then move to the exercises below, that will ask you to comment on what you observe in different scenarios. 

Parameters are as follows:
* $a_{ij}$: components of the lattice vectors
* $\phi$: rigid rotation of the lattice (does it have an impact on the diffraction?)
* $s_{1,2}$: fractional coordinates of the second atom of the basis (first atom is in (0,0))
* $f_{1,2}$: atomic form factors for the two atoms in the basis (roughly take it to be the atomic number)
* $\lambda$: wavelength of the scattering radiation

In [ ]:
# set upt the code widget window
def diffraction_peaks(
    basis: list[list],
    atomic_ff: float,
    reciprocal_b1: list,
    reciprocal_b2: list,
    wavelength: float,
):
    """
    Computes the list of peaks for a lattice with a given (real-space) basis 
    and reciprocal lattice, and for a given wavelength of the incoming radiation.

    :param basis: list of N 2D vectors corresponding to the (real space!) position of the basis atoms
    :param atomic_form_factors: atomic form factors of atoms in lattice, array of length N
    :param reciprocal_b1, reciprocal_b2:  reciprocal lattice vectors
    :param wavelength: wavelength of the incoming radiation

    :return: The list of diffraction peaks, as [h, l, theta, intensity]
    """,
    import numpy as np

    def compute_absolute_structure_factor(sj, fj, G):
        # sj: atomic basis (n_basis x 2)
        # fj: form factors (n_basis)
        # G: reciprocal lattice vectors (2D)
        return np.abs(fj @ np.exp(-1j * sj @ G))

    # wave number (modulus of the incoming wavevector)
    k = np.pi * 2 / wavelength
    # determine the range of reciprocal lattice vectors that could give rise to permissible reflections
    if reciprocal_b1 @ reciprocal_b1 != 0:
        n1 = int((k * 2) / np.sqrt(reciprocal_b1 @ reciprocal_b1)) + 1
    else:
        n1 = 0
    if reciprocal_b2 @ reciprocal_b2 != 0:
        n2 = int((k * 2) / np.sqrt(reciprocal_b2 @ reciprocal_b2)) + 1
    else:
        n2 = 0

    # allocated space for the list of peaks
    lpeaks = []
    for v1 in range(-n1, n1 + 1):
        for v2 in range(-n2, n2 + 1):
            # reciprocal lattice vector
            G = reciprocal_b1 * v1 + reciprocal_b2 * v2

            # theta (from 2theta geometry)
            sin_theta = np.sqrt(G @ G) / (2 * k)
            if (
                sin_theta > 1
            ):  # discards reflections that fall outside of the permissible range
                continue
            theta = np.arcsin(sin_theta)
            # structure factor
            absolute_structure_factor = compute_absolute_structure_factor(
                basis, atomic_ff, G
            )
            lpeaks.append([v1, v2, theta, absolute_structure_factor**2])
    return np.asarray(lpeaks)

In [ ]:
from ipywidgets import HTML, HBox, Layout
import functools


def rot2d(angle):
    return np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])


def plot_diffraction(code_demo: CodeExercise):
    # ,code_input,visualizers):
    cue_figure = code_demo.cue_outputs[0]
    axes = cue_figure.figure.get_axes()
    a11, a12, a21, a22, s1, s2, f1, f2, phi, wavelength = code_demo.parameters.values()
    rot_phi = rot2d(phi)
    a1 = np.array([a11, a12]) @ rot_phi
    a2 = np.array([a21, a22]) @ rot_phi
    basis = np.asarray([0 * a1, s1 * a1 + s2 * a2])
    plot_lattice(axes[0], a1, a2, basis=basis, alphas=[f1 / 40, f2 / 40], c="red")
    axes[0].set_title("real space")
    axes[0].set_xlim(-5, 5)
    axes[0].set_ylim(-5, 5)
    axes[0].set_xlabel("$x$ / Å")
    axes[0].set_ylabel("$y$ / Å")

    b1, b2 = reciprocal_lattice_vectors(a1, a2)

    dpeaks = code_demo.run_code(basis, np.asarray([f1, f2]), b1, b2, wavelength)

    twotheta_grid = np.linspace(0, 180, 720)
    dp_grid = np.zeros(len(twotheta_grid))
    for _, _, t, f2 in dpeaks:
        dp_grid += np.exp(-((twotheta_grid - 2 * t * 180 / np.pi) ** 2) / 0.5) * f2

    axes[1].clear()
    # we plot two theta
    axes[1].plot(twotheta_grid, dp_grid, "b-")

    axes[1].set_xlim(0, 180)
    axes[1].set_xlabel("$2\\theta$ / degree °")
    axes[1].set_title("Diffraction pattern")
    axes[1].set_ylabel("Intensity $|F(\mathbf{k}(\\theta))|$")

    axes[0].set_aspect("equal")
    axes[1].set_aspect(150 / np.max(dp_grid))
    cue_output = code_demo.cue_outputs[1]
    global reflection_table_html
    cue_output.clear_output(wait=True)
    with cue_output:
        header = """
                      v1 / Å 
                      v2 / Å 
                      2θ / degree ° 
                      |FG|2 
                    """
        # cleans up peak info for displaying
        tpeaks = []
        for d in dpeaks[np.argsort(dpeaks[:, 2])]:
            tpeaks.append(
                [
                    int(d[0]),
                    int(d[1]),
                    np.round(2 * d[2] * 180 / np.pi, 2),
                    np.round(d[3], 1),
                ]
            )
        reflection_table_html.value = array_to_html_table(tpeaks, header)


def array_to_html_table(numpy_array, header):
    rows = ""
    for i in range(len(numpy_array)):
        rows += (
            "<tr>"
            + functools.reduce(
                lambda x, y: x + y,
                map(lambda x: "<td>" + str(x) + "</td>", numpy_array[i]),
            )
            + "</tr>"
        )

    return "<table>" + header + rows + "</table>"


reflection_table_html = HTML(value=f"dpeaks")

reflection_table = HBox(layout=Layout(width="99%", height="250px", overflow_y="auto"))
reflection_table.children += (reflection_table_html,)
reflection_table_output = CueOutput()
with reflection_table_output:
    display(reflection_table)

In [ ]:
diffraction_figure, _ = plt.subplots(1, 2, figsize=(7.5, 3.8), tight_layout=True)
diffraction_output = CueFigure(diffraction_figure)

ex12_parameters = dict(
    a11=FloatSlider(1.0, min=-4, max=4, step=0.1, description=r"$a_{11} / Å$"),
    a12=FloatSlider(0.0, min=-4, max=4, step=0.1, description=r"$a_{12} / Å$"),
    a21=FloatSlider(0.0, min=-4, max=4, step=0.1, description=r"$a_{21} / Å$"),
    a22=FloatSlider(2.0, min=-4, max=4, step=0.1, description=r"$a_{22} / Å$"),
    s1=FloatSlider(0.25, min=0.01, max=0.99, step=0.01, description=r"$s_1$"),
    s2=FloatSlider(0.75, min=0.01, max=0.99, step=0.01, description=r"$s_2$"),
    f1=IntSlider(10.0, 1.0, 40.0, 1, description=r"$f_{1}$"),
    f2=IntSlider(30.0, 0, 40.0, 1, description=r"$f_{2}$"),
    phi=FloatSlider(0.0, min=0.0, max=2 * np.pi, step=0.05, description=r"$\phi$"),
    wavelength=FloatSlider(
        1.0, min=0.2, max=2, step=0.05, description=r"$\lambda$ / Å"
    ),
)


ex12_code_demo = CodeExercise(
    code=diffraction_peaks,
    parameters=ex12_parameters,
    cue_outputs=[diffraction_output, reflection_table_output],
    update_func=plot_diffraction,
    update_mode="manual",
)

display(ex12_code_demo)
ex12_code_demo.run_update()

Set $f_2$ to zero (so that effectively this becomes a lattice with a single atom) and set the unit cell to be a $1\times 1.2$ rectangle. Set the wavelength to 1. Observe how the position and intensity of the peaks change when you change the dimensions of the lattice.

In [ ]:
ex12a_description = """What happens if you set the off-diagonal term $a_{12}$ to be (slightly)
different from zero? What happens if you set the two lattice vectors to be
orthogonal and equal in length?"""
ex12a_answer = "Enter your answer here"

### BEGIN SOLUTION
ex12a_answer = """If one changes the shape of the lattice, all the peak positions move around.
Increasing the cell size brings peaks closer to theta=0, and more peaks become
visible. Decreasing the cell size increase theta and reduces the number of
peaks.  If you change the off-diagonal terms to a non-zero value, the peaks at
81 and 153 degrees - that are composed of four degenerate peaks - split into
two pairs of peaks. The hk,-h-k and h-k,-hk peaks remains (because the lattice
has inversion symmetry). If all lattice vectors are made all equal, the level
of degeneracy increases: now also hk and kh peaks have the same angle."""
### END SOLUTION


ex12a_exercise = TextExercise(
    ex12a_answer,
    answer_registry=answer_registry,
    answer_key="ex12a",
    exercise_description=ex12a_description,
    exercise_title="Exercise 12a",
)
display(ex12a_exercise)

In [ ]:
ex12a_description = """What happens if you set the off-diagonal term $a_{12}$ to be (slightly)
different from zero? What happens if you set the two lattice vectors to be
orthogonal and equal in length?"""
ex12a_answer = "Enter your answer here"

### BEGIN SOLUTION
ex12a_answer = """If one changes the shape of the lattice, all the peak positions move around.
Increasing the cell size brings peaks closer to theta=0, and more peaks become
visible. Decreasing the cell size increase theta and reduces the number of
peaks.  If you change the off-diagonal terms to a non-zero value, the peaks at
81 and 153 degrees - that are composed of four degenerate peaks - split into
two pairs of peaks. The hk,-h-k and h-k,-hk peaks remains (because the lattice
has inversion symmetry). If all lattice vectors are made all equal, the level
of degeneracy increases: now also hk and kh peaks have the same angle."""
### END SOLUTION


ex12a_exercise = TextExercise(
    ex12a_answer,
    answer_registry=answer_registry,
    answer_key="ex12a",
    exercise_description=ex12a_description,
    exercise_title="Exercise 12a",
)
display(ex12a_exercise)

Set $f_1=30$, $f_2=10$, and set the unit cell to be a $1\times 1.2$ rectangle. Set the fractional coordinates of the second atom to be $(0.5,0.5)$. Set the wavelength to 1. Observe how the position and intensity of the peaks change when you change the form factor of the second atom from $0$ to be equal to $f_1$.


In [ ]:
ex12b_description = """Do the peak positions change? How do you explain the change in intensity?  What
happens if (keeping the form factors equal) you change one of the fractional
coordinates to be different from $0.5$? And if you change both coordinates? How
do you explain this observation?"""
ex12b_answer = "Enter your answer here"

### BEGIN SOLUTION
ex12b_answer = """The peak positions is completely independent on the form factor or the position
of the second atom in the basis. However the intensity changes greatly. If f1
is set to be equal to f2 some of the the peaks disappear. This is because the
structure becomes a face-centered rectangular, and has actually a smaller
primitive cell, with longer reciprocal-lattice vectors. Moving the central atom
away from the cell center restores some of the intensity on the peaks that had
disappeared. Disappearance of other peaks can also be seen if the second atom
is placed in (0.5,0) or (0,0.5)"""
### END SOLUTION


ex12b_exercise = TextExercise(
    ex12b_answer,
    answer_registry=answer_registry,
    answer_key="ex12b",
    exercise_description=ex12b_description,
    exercise_title="Exercise 12b",
)
display(ex12b_exercise)

Set $f_1=30$, $f_2=10$, and set the unit cell to be a $1\times 1$ square. Set the fractional coordinates of the second atom to be $(0.5,0.5)$. Set the wavelength to 1. Observe how the position and intensity of the peaks change when you change the form factor of the second atom from $0$ to be $f_2=f_1=30$.


In [ ]:
ex12c_description = """How many peaks are left when $f_2=f_1$? How do you explain this? You can obtain
exactly the same diffraction pattern by setting $f_2=0$ and changing the
lattice parameters. What lattice parameters should you use?"""
ex12c_answer = "Enter your answer here"

### BEGIN SOLUTION
ex12c_answer = """Having a square lattice reduces even further the number of peaks - because 0h
and h0 peaks become degenerate. When the two atoms are set to the same form
factor, the intensity of the 01 and 10 peaks decays to zero, and we have a
single peak at 2theta = 90 degrees: we have now effectively a much smaller unit
cell, with a size that is equal to half the diagonal of the initial 1x1 cell.
Indeed we can obtain exactly the same diffraction pattern by setting f2=0 and
the axes size to sqrt(2)/2~0.707"""
### END SOLUTION


ex12c_exercise = TextExercise(
    ex12c_answer,
    answer_registry=answer_registry,
    answer_key="ex12c",
    exercise_description=ex12c_description,
    exercise_title="Exercise 12c",
)
display(ex12c_exercise)